In [0]:

# DATA CENTER IoT - SAFE DATA RETENTION
# Raw + Bronze + Silver + Gold + Reports
# Other project data remains untouched

from datetime import datetime, timedelta, timezone
from pyspark.sql.functions import col

IST = timezone(timedelta(hours=5, minutes=30))

base = "/Volumes/workspace/iot_db/data"

# True = Preview only
# False = Delete expired data
DRY_RUN = False

# Retention settings
RAW_DAYS = 3
BRONZE_DAYS = 7
SILVER_DAYS = 7
GOLD_DAYS = 30
REPORT_DAYS = 7

# VACUUM configuration
# 168 hours = 7 days of Delta file history
VACUUM_HOURS = 168

# Keep False until separately validated
ENABLE_VACUUM = False

today = datetime.now(IST).date()

print("=" * 55)
print("DATA CENTER RETENTION STARTED")
print("Today (IST):", today)
print("DRY RUN:", DRY_RUN)
print("VACUUM Enabled:", ENABLE_VACUUM)
print("=" * 55)


# STEP 1: Safely list folders
def safe_ls(path):
    try:
        return dbutils.fs.ls(path)
    except Exception as e:
        print("Skipping unavailable path:", path)
        print("Reason:", e)
        return []


# STEP 2: Raw CSV retention
raw_path = f"{base}/dc_raw"
raw_cutoff = today - timedelta(days=RAW_DAYS)

print("\n--- RAW CSV RETENTION ---")
print("Cutoff:", raw_cutoff)

for folder in safe_ls(raw_path):

    if not folder.isDir():
        continue

    if not folder.name.startswith("process_date="):
        continue

    date_text = folder.name.replace(
        "process_date=", ""
    ).strip("/")

    try:
        folder_date = datetime.strptime(
            date_text, "%Y-%m-%d"
        ).date()
    except ValueError:
        print("Skipping unknown folder:", folder.path)
        continue

    if folder_date < raw_cutoff:

        print("Expired Raw folder:", folder.path)

        if not DRY_RUN:
            dbutils.fs.rm(folder.path, True)
            print("Deleted:", folder.path)


# STEP 3: Delta retention
def clean_delta(path, days):

    cutoff = today - timedelta(days=days)

    print("\n--- DELTA RETENTION ---")
    print("Path:", path)
    print("Cutoff:", cutoff)

    df = spark.read.format("delta").load(path)

    expired_count = (
        df.filter(col("process_date") < str(cutoff))
        .count()
    )

    print("Expired records:", expired_count)

    if expired_count > 0:

        if DRY_RUN:
            print("DRY RUN - no records deleted")

        else:
            spark.sql(f"""
                DELETE FROM delta.`{path}`
                WHERE process_date < DATE '{cutoff}'
            """)

            print("Expired records logically deleted")

    else:
        print("No expired records")

    # VACUUM only when explicitly enabled
    if ENABLE_VACUUM:

        if DRY_RUN:
            print("VACUUM preview only")

            spark.sql(f"""
                VACUUM delta.`{path}`
                RETAIN {VACUUM_HOURS} HOURS
                DRY RUN
            """).show(truncate=False)

        else:
            spark.sql(f"""
                VACUUM delta.`{path}`
                RETAIN {VACUUM_HOURS} HOURS
            """)

            print("VACUUM completed:", path)


clean_delta(f"{base}/dc_bronze", BRONZE_DAYS)
clean_delta(f"{base}/dc_silver", SILVER_DAYS)
clean_delta(f"{base}/dc_gold", GOLD_DAYS)


# STEP 4: Gold SQL table
gold_table = "workspace.iot_db.gold_dc_summary"

gold_cutoff = today - timedelta(days=GOLD_DAYS)

print("\n--- GOLD SQL TABLE RETENTION ---")
print("Table:", gold_table)
print("Cutoff:", gold_cutoff)

gold_expired_count = (
    spark.table(gold_table)
    .filter(col("process_date") < str(gold_cutoff))
    .count()
)

print("Expired Gold records:", gold_expired_count)

if gold_expired_count > 0:

    if DRY_RUN:
        print("DRY RUN - no records deleted")

    else:
        spark.sql(f"""
            DELETE FROM {gold_table}
            WHERE process_date < DATE '{gold_cutoff}'
        """)

        print("Expired Gold table records deleted")


# STEP 5: CSV report retention
output_path = f"{base}/output"
report_cutoff = today - timedelta(days=REPORT_DAYS)

print("\n--- CSV REPORT RETENTION ---")
print("Cutoff:", report_cutoff)

for file in safe_ls(output_path):

    if file.isDir():
        continue

    if not file.name.startswith("dc_gold_summary_"):
        continue

    try:
        file_date = datetime.strptime(
            file.name,
            "dc_gold_summary_%Y-%m-%d.csv"
        ).date()
    except ValueError:
        continue

    if file_date < report_cutoff:

        print("Expired report:", file.path)

        if not DRY_RUN:
            dbutils.fs.rm(file.path)
            print("Deleted:", file.path)


# STEP 6: Shared logs protection
print("\n--- LOG RETENTION ---")
print("Shared logs folder skipped")
print("Other project logs remain untouched")


# STEP 7: Final status
print("\n" + "=" * 55)
print("DATA CENTER RETENTION COMPLETED")

if DRY_RUN:
    print("DRY RUN ONLY - NO DATA DELETED")
else:
    print("EXPIRED DATA CLEANUP COMPLETED")

if not ENABLE_VACUUM:
    print("VACUUM DISABLED - NO PHYSICAL DELTA CLEANUP")

print("=" * 55)


DATA CENTER RETENTION STARTED
Today (IST): 2026-10-24
DRY RUN: True
VACUUM Enabled: True

--- RAW CSV RETENTION ---
Cutoff: 2026-10-21
Expired Raw folder: dbfs:/Volumes/workspace/iot_db/data/dc_raw/process_date=2026-10-19/
Expired Raw folder: dbfs:/Volumes/workspace/iot_db/data/dc_raw/process_date=2026-10-20/

--- DELTA RETENTION ---
Path: /Volumes/workspace/iot_db/data/dc_bronze
Cutoff: 2026-10-17
Expired records: 0
No expired records
VACUUM preview only
+----+
|path|
+----+
+----+


--- DELTA RETENTION ---
Path: /Volumes/workspace/iot_db/data/dc_silver
Cutoff: 2026-10-17
Expired records: 0
No expired records
VACUUM preview only
+----+
|path|
+----+
+----+


--- DELTA RETENTION ---
Path: /Volumes/workspace/iot_db/data/dc_gold
Cutoff: 2026-09-24
Expired records: 0
No expired records
VACUUM preview only
+----+
|path|
+----+
+----+


--- GOLD SQL TABLE RETENTION ---
Table: workspace.iot_db.gold_dc_summary
Cutoff: 2026-09-24
Expired Gold records: 0

--- CSV REPORT RETENTION ---
Cutoff: 20